# English-Luganda Translator - ML Pipeline on Colab

**GPU-Accelerated ML Workflow**

This notebook:
- Clones from GitHub
- Loads 50,000+ dataset pairs
- Trains on Tesla T4 GPU
- Calculates BLEU score

## STEP 1: Install Packages

In [2]:
import subprocess, sys
print("="*80)
print("ENGLISH-LUGANDA TRANSLATOR - COLAB ML PIPELINE")
print("="*80)
print("\n[STEP 1: Installing packages]\n")
packages = ["torch", "transformers", "datasets", "pandas", "scikit-learn", "sacrebleu"]
print("Installing packages...")
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
    print(f"  ✓ {pkg}")
print("\n✅ Done")

ENGLISH-LUGANDA TRANSLATOR - COLAB ML PIPELINE

[STEP 1: Installing packages]

Installing packages...
  ✓ torch
  ✓ transformers
  ✓ datasets
  ✓ pandas
  ✓ scikit-learn
  ✓ sacrebleu

✅ Done


## STEP 2: Clone from GitHub

In [3]:
import os, subprocess
print("\n[STEP 2: Cloning from GitHub]\n")

REPO_PATH = "/content/ENGLISH-LUGANDA-TRANSLATOR"

# Clone if needed
if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", "https://github.com/ndagirenairah/ENGLISH-LUGANDA-TRANSLATOR.git", REPO_PATH], check=True)
    print("✅ Repository cloned")
else:
    os.chdir(REPO_PATH)
    subprocess.run(["git", "pull"], check=False)
    print("✅ Repository updated")

# Check if we need to navigate to nested folder
nested_path = os.path.join(REPO_PATH, "ENGLISH-LUGANDA-TRANSLATOR")
if os.path.exists(nested_path) and os.path.exists(os.path.join(nested_path, "src")):
    os.chdir(nested_path)
    print(f"✅ Using nested folder: {os.getcwd()}")
elif os.path.exists(os.path.join(REPO_PATH, "src")):
    os.chdir(REPO_PATH)
    print(f"✅ Using root folder: {os.getcwd()}")
else:
    print(f"❌ Could not find src/ folder")
    print(f"   REPO_PATH: {os.listdir(REPO_PATH)}")



[STEP 2: Cloning from GitHub]

✅ Repository cloned
✅ Using root folder: /content/ENGLISH-LUGANDA-TRANSLATOR


## STEP 3: Load Datasets

In [4]:
print("\n[STEP 3: Loading datasets]\n")
import sys
sys.path.insert(0, "src")
from load_data import load_all_datasets, get_dataset_statistics
combined_df = load_all_datasets()
stats = get_dataset_statistics(combined_df)
print(f"\n✅ Loaded {stats['total_samples']:,} samples")
print(f"   English avg: {stats['avg_english_length']:.1f}")
print(f"   Luganda avg: {stats['avg_luganda_length']:.1f}")


[STEP 3: Loading datasets]


  STEP 1: LOADING DATASETS

📂 Loading kambale...
   ✅ Loaded 50,012 valid pairs | Source: kambale_train.csv

📂 Loading cultural...
   ✅ Loaded 12 valid pairs | Source: cultural_training.csv

📂 Loading jw300...
   ✅ Loaded 15 valid pairs | Source: jw300_parallel.csv

📂 Loading makerere...
   ✅ Loaded 15 valid pairs | Source: makerere_nlp.csv

📂 Loading sunbird...
   ✅ Loaded 18 valid pairs | Source: sunbird_salt.csv

  DATASET SUMMARY

📊 Total samples loaded: 50,072

📈 Breakdown by source:
   kambale_train.csv              50,012 ( 99.9%)
   sunbird_salt.csv                   18 (  0.0%)
   jw300_parallel.csv                 15 (  0.0%)
   makerere_nlp.csv                   15 (  0.0%)
   cultural_training.csv              12 (  0.0%)

✅ Loaded 50,072 samples
   English avg: 53.7
   Luganda avg: 57.1


## STEP 4: Preprocess & Split

In [5]:
print("\n[STEP 4: Preprocessing]\n")
from preprocess import preprocess_and_split, save_splits
train_df, val_df, test_df = preprocess_and_split(combined_df)
save_splits(train_df, val_df, test_df)
print(f"✅ Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")


[STEP 4: Preprocessing]


  STEP 2: PREPROCESSING DATA

🧹 Cleaning text...
   ✅ 50,072 valid pairs

📊 Creating train/val/test splits...
   Train: 80% | Val: 10% | Test: 10%
   Train: 40,056 | Val: 5,008 | Test: 5,008

💾 Saving splits...
   ✅ Saved to /content/ENGLISH-LUGANDA-TRANSLATOR/data/processed/
      - train.csv (40,056 samples)
      - val.csv (5,008 samples)
      - test.csv (5,008 samples)
✅ Train: 40,056 | Val: 5,008 | Test: 5,008


## STEP 5: Train Model (8-12 min on GPU)

In [6]:
print("\n[STEP 5: Training]\n")
import torch, pathlib
print(f"GPU: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  Device: {torch.cuda.get_device_name(0)}")
train_file = pathlib.Path("src/train.py")
if train_file.exists():
    code = train_file.read_text()
    code = code.replace("        tokenizer=tokenizer,", "")
    train_file.write_text(code)
    print("✅ Applied hotfix\n")
print("="*80)
print("STARTING TRAINING")
print("="*80)
from train import main as train_main
try:
    model, tokenizer = train_main()
    print("\n✅ Training complete!")
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()


[STEP 5: Training]

GPU: True
  Device: Tesla T4
✅ Applied hotfix

STARTING TRAINING

  STEP 3: TRAINING MODEL

📁 Loading training data...
   Train: 40,056 | Val: 5,008

🤖 Loading model: Helsinki-NLP/opus-mt-en-mul


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/790k [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_positions.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].



✅ Training complete!
   Training loss: 0.7578
   Time elapsed: 31.5 minutes

💾 Saving model and tokenizer...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

   ✅ Saved to /content/ENGLISH-LUGANDA-TRANSLATOR/models/trained_model

  TRAINING COMPLETE

✅ Model trained successfully!

   Next step: python src/4_evaluate.py

✅ Training complete!


## STEP 6: Evaluate

In [7]:
print("\n[STEP 6: Evaluating]\n")
from evaluate import main as eval_main
try:
    eval_main()
    print("\n✅ Evaluation complete!")
except Exception as e:
    print(f"❌ Error: {e}")
    import traceback
    traceback.print_exc()


[STEP 6: Evaluating]


  STEP 4: EVALUATING MODEL

📁 Loading test data...
   Test samples: 5008

🤖 Loading trained model...
📂 Loading model from /content/ENGLISH-LUGANDA-TRANSLATOR/models/trained_model


/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



📊 Evaluating on test set...
   Test samples: 5008
   Generating translations...
      [5008/5008]
   Calculating BLEU score...

   ✅ BLEU Score: 20.69

   📈 Additional Metrics:
      Average prediction length: 6.8 tokens
      Average reference length: 7.2 tokens

SAMPLE PREDICTIONS

Sample 1:
  Reference: Bwe baayawukana yasalawo okukuza yekka abaana be.
  Predicted: Oluvannyuma lw'okuwulizibwa yasalawo okukuza abaana be yekka.

Sample 2:
  Reference: Uganda kati erina ssaabalamuzi omuggya.
  Predicted: Uganda kati erina ssaabalamuzi omupya.

Sample 3:
  Reference: Atwalira ebirabo by'emmere bisatu mu kitundu kye enkoko.
  Predicted: Awa enkokoko eri emirimbe esatu mu kitundu.

Sample 4:
  Reference: Bbanka yagaana okumuwola ssente.
  Predicted: Bbanka yagaana ssente ze yasaba.

Sample 5:
  Reference: Akozze nnyo omwaka guno.
  Predicted: Afiiriddwa nnyo omwaka guno.

💾 Saving results...
   ✅ Saved to /content/ENGLISH-LUGANDA-TRANSLATOR/outputs/evaluation_results.json
   ✅ Prediction

## STEP 7: Results

In [8]:
import json
from pathlib import Path
eval_file = Path("outputs/evaluation_results.json")
if eval_file.exists():
    with open(eval_file) as f:
        results = json.load(f)
    print("\n" + "="*80)
    print("FINAL RESULTS")
    print("="*80)
    print(f"\nBLEU Score: {results['bleu_score']:.2f}")
    print(f"Test samples: {results['num_test_samples']}")


FINAL RESULTS

BLEU Score: 20.69
Test samples: 5008


In [9]:
import shutil
from pathlib import Path
from google.colab import files

model_path = Path("models/trained_model")
if model_path.exists():
    shutil.make_archive("trained_model", "zip", model_path.parent, model_path.name)
    print("✅ Downloading trained_model.zip...")
    files.download("trained_model.zip")

✅ Downloading trained_model.zip...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Pipeline Complete!